In [0]:
pip install langchain-google-genai python-dotenv tabulate delta-spark --quiet

In [0]:
%skip
%restart_python

In [0]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
# from pyspark.sql import SparkSession

In [0]:
df_btc_weekly = (spark.table("workspace.gold.tb_bitcoin_weekly")
           .orderBy("DT_WEEK_START", ascending=False)
           .limit(8)
           )
df_btc_weekly.display()

In [0]:
cols = df_btc_weekly.columns
rows = df_btc_weekly.collect()

header = "| " + " | ".join(cols) + " |"
separator = "| " + " | ".join(["---"] * len(cols)) + " |"
body = "\n".join("| " + " | ".join(str(v) for v in r) + " |" for r in rows)
df_weekly_markdown = "\n".join([header, separator, body])

In [0]:
print(df_weekly_markdown)

In [0]:
#read prompt from prompts folder
with open('PROMPTS/analysis_agent.txt') as file:
    prompt_file = file.read()
#langchain
# llm = ChatGoogleGenerativeAI(model_name="models/chat-bison", temperature=0.0


# substitui o placeholder pelos dados
prompt = prompt_file.replace('{dados_semanal}', df_weekly_markdown)

# chama o Gemini
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash-lite',
    google_api_key=os.getenv('GEMINI_API_KEY')
)

resposta = llm.invoke(prompt)
analise = resposta.content

print(analise)